#### Benchmark subtract_z_motion_patches
Test with T-series recorded with no calcium activity, only z motion.  
  
T-series data: `Scnn1aAi14_B1M0/04222024/TSeries-04222024-1047-001`
`C57_O1M1/09182023/TSeries-09182023-1503-003`  
Z-series: `Scnn1aAi14_B1M0/04222024/ZSeries-04222024-1047-001`
`C57_O1M1/09182023/ZSeries-09182023-1503-003`

In [ ]:
# !pip install plotly

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import mesmerize_core as mc

import sys
sys.path.append("..")  # Add above scripts directory to python modules path.
import pipeline_mcorr_cnmf as pmc
import compute_zcorr as cz

path_file = '/home/wanglab/code/imaging_analysis/Analysis_2P/test/Paths_zmotion_lin2.json'
# '/home/wanglab/code/imaging_analysis/Analysis_2P/test/Paths_zmotion_lin.json'
# 
# 'D:/Code/Analysis_2P/test/Paths_zmotion_win.json'

with open(path_file) as f:
    paths = json.load(f)
    
params_files = paths['params_files']
with open(params_files[0]) as f:
    parameters = json.load(f)

zstack_path = paths['zstack_paths']
zstack_path = Path(zstack_path[0])
z_params_files = paths['z_params_files']
with open(z_params_files[0]) as f:
    z_parameters = json.load(f)
    
z_shifted_file = z_parameters['zstack_shift']['file_name']

# Get the motion correction parameters
parameters_mcorr = parameters['params_mcorr']

In [ ]:
# Run the motion correction 
data_paths = paths['data_paths']
export_paths = paths['export_paths']
regex_pattern='*_Ch2_*.ome.tif'

export_path= Path(export_paths[0])
# If export path does not exist, create it
if not export_path.exists():
    os.makedirs(export_path)
# Set the parent raw data path
mc.set_parent_raw_data_path(export_path) 

# Run the x/y motion correction
# batch_path, index, mcorr_movie_path = pmc.run_mcorr(data_paths, export_path, parameters_mcorr, regex_pattern)
# mcorr_movie_path = export_path / 'e28adbc4-f6b0-40aa-a251-3bef430ae77f/e28adbc4-f6b0-40aa-a251-3bef430ae77f-cat_tiff_bt_els__d1_765_d2_765_d3_1_order_F_frames_960.mmap'
# mcorr_movie_path = export_path / 'c8a348c2-d6da-4196-bb23-0a84371b721a/c8a348c2-d6da-4196-bb23-0a84371b721a-cat_tiff_bt_els__d1_765_d2_765_d3_1_order_F_frames_960.mmap'
mcorr_movie_path = export_path / 'f31ed550-5898-4608-9b55-7dd1fb5610f9/f31ed550-5898-4608-9b55-7dd1fb5610f9-cat_tiff_bt_els__d1_765_d2_765_d3_1_order_F_frames_960.mmap'

# Concatenated tiff file(s) converted to uint16 range, from uint12 range, in pipeline_mcorr_cnmf > run_mcorr -> bruker_concat_tif > convert_to_bigtiff
# https://vscode.dev/github/pseudomanu/Analysis_2P/blob/dev_zcorr/Mesmerize/bruker_concat_tif.py#L195

### Compute z-correlation
`compute_zcorrel` returns the z-correlation between the T-series and the Z-series, with x and y shifts, and the estimated z position of the T-series' frames.  

In [ ]:
#  Old code - Suite2p based
# import compute_zcorr as cz
z_correlation=cz.compute_zcorrel_suite2p(zstack_path / z_shifted_file, mcorr_movie_path)

In [ ]:
# Shift zstack
if not os.path.exists(zstack_path / z_shifted_file):
    shift_zstack_path=cz.shift_zstack(z_parameters['zstack_shift'], zstack_path, z_shifted_file)
else:
    shift_zstack_path=zstack_path / z_shifted_file

#  Shifted z-stack values converted to uint16 range, from uint12 range, in compute_zcorr > shift_zstack
#  https://vscode.dev/github/pseudomanu/Analysis_2P/blob/dev_zcorr/Mesmerize/compute_zcorr.py#L120

# Compute z-correlation
mesmerize_path = mcorr_movie_path.parents[1] # get mesmerize directory where the zcorr file will be saved
if not os.path.exists(mesmerize_path / "z_correlation.npz"):
    z_correlation=cz.compute_zcorrel(zstack_path / z_shifted_file, mcorr_movie_path)
    # np.save(mesmerize_path / "z_correlation.npy", z_correlation)   
else:
    z_correlation=np.load(mesmerize_path / "z_correlation.npz")
  
# "Old" suite2p-based code    
# futures.ThreadPoolExecutor: 960 frames processed in 212.00 sec. (3mn 32s)
# joblib.Parallel: 960 frames processed in 120.26 sec. (2mn 0s)
# New code, simpler z-correl
# futures.ThreadPoolExecutor: 960 frames processed in ~600 sec. (10mn)
# joblib.Parallel: 960 frames processed in 496.22 sec. (8mn 16s)

In [ ]:
mesmerize_path = mcorr_movie_path.parents[1]
# z_correlation = np.load(mesmerize_path / "z_correlation_bu.npz")
# z_pos = z_correlation['zpos']
# z_correlation_ThreadPoolExecutor = np.load(mesmerize_path / "z_correlation_ThreadPoolExecutor.npz")
# z_pos_ThreadPoolExecutor = z_correlation_ThreadPoolExecutor['zpos']
z_correlation_Parallel = np.load(mesmerize_path / "z_correlation_Parallel.npz")
z_pos_Parallel = z_correlation_Parallel['zpos']
z_correlation_Parallel_2ndround = np.load(mesmerize_path / "z_correlation_Parallel_2ndround.npz")
z_pos_Parallel_2 = z_correlation_Parallel_2ndround['zpos']
# z_correlation_new_TPE= np.load(mesmerize_path / "z_correlation_new_TPE.npz")
# z_pos_new_TPE = z_correlation_new_TPE['zpos']
# z_correlation_new2_TPE= np.load(mesmerize_path / "z_correlation_new2_TPE.npz")
# z_pos_new_TPE = z_correlation_new2_TPE['zpos']
# z_correlation_new2_TPE_2ndround= np.load(mesmerize_path / "z_correlation_new2_TPE_replica.npz")
# z_pos_new2 = z_correlation_new2_TPE_2ndround['zpos']
z_correlation_new2_Parallel= np.load(mesmerize_path / "z_correlation_new2_Parallel.npz")
z_pos_new_Parallel = z_correlation_new2_Parallel['zpos']

#  Plot the three z_pos on top of each other
import plotly.graph_objects as go
fig = go.Figure()
# fig.add_trace(go.Scatter(y=z_pos, mode='lines', name='z_pos'))
# fig.add_trace(go.Scatter(y=z_pos_ThreadPoolExecutor, mode='lines', name='z_pos_ThreadPoolExecutor'))
# fig.add_trace(go.Scatter(y=z_pos_Parallel, mode='lines', name='z_pos_Parallel'))
# fig.add_trace(go.Scatter(y=z_pos_new_TPE, mode='lines', name='z_pos_new_TPE'))
fig.add_trace(go.Scatter(y=z_pos_new_Parallel, mode='lines', name='z_pos_Parallel_new_code'))
fig.add_trace(go.Scatter(y=z_pos_Parallel, mode='lines', name='z_pos_Parallel_Suite2p'))
fig.show()
 

In [ ]:
print(z_pos_new_Parallel[:20])
print(z_pos_Parallel[:20])
z_pos_Parallel[:20] - z_pos_new_Parallel[:20]

In [ ]:
# Plot the max diff between the three traces
# Calculate the absolute differences between the traces
# Convert data to a signed integer type
z_pos = z_pos.astype(np.int32)
z_pos_ThreadPoolExecutor = z_pos_ThreadPoolExecutor.astype(np.int32)
z_pos_Parallel = z_pos_Parallel.astype(np.int32)

# Now calculate the differences
diff1 = np.abs(z_pos - z_pos_ThreadPoolExecutor)
diff2 = np.abs(z_pos - z_pos_Parallel)
diff3 = np.abs(z_pos_ThreadPoolExecutor - z_pos_Parallel)

# Calculate the maximum difference at each frame
zpos_diff = np.max([diff1, diff2, diff3], axis=0)

# Plot the max diff
import matplotlib.pyplot as plt
plt.plot(zpos_diff)
plt.xlabel('Frame')
plt.ylabel('Max difference')
plt.title('Max difference between z_pos traces')
plt.show()


In [ ]:
# Get xshift and yshift data from z_pos_Parallel and z_pos_new_Parallel. Compare values over frames
xshift_Parallel = z_correlation_Parallel['xshift']
xshift_new_Parallel = z_correlation_new2_Parallel['xshift']
yshift_Parallel = z_correlation_Parallel['yshift']
yshift_new_Parallel = z_correlation_new2_Parallel['yshift']

# Filter xshift_Parallel (41, 960) with z_pos_Parallel (960,) as index
xshift_Parallel_zpos = np.array([xshift_Parallel[i, 0] for i in z_pos_Parallel])
xshift_new_Parallel_zpos = np.array([xshift_new_Parallel[i, 0] for i in z_pos_new_Parallel])
yshift_Parallel_zpos = np.array([yshift_Parallel[i, 0] for i in z_pos_Parallel])
yshift_new_Parallel_zpos = np.array([yshift_new_Parallel[i, 0] for i in z_pos_new_Parallel])

In [ ]:
# Find max diff between xshift_Parallel_zpos and xshift_new_Parallel_zpos, and between yshift_Parallel_zpos and yshift_new_Parallel_zpos
xshift_diff = np.max(np.abs(xshift_Parallel_zpos - xshift_new_Parallel_zpos))
yshift_diff = np.max(np.abs(yshift_Parallel_zpos - yshift_new_Parallel_zpos))
print(f'Max diff between xshift_Parallel_zpos and xshift_new_Parallel_zpos: {xshift_diff}')
print(f'Max diff between yshift_Parallel_zpos and yshift_new_Parallel_zpos: {yshift_diff}')
#  Find first frame where the difference is greater than 0 in xshift and yshift
xshift_diff_frame = np.where(np.abs(xshift_Parallel_zpos - xshift_new_Parallel_zpos) > 0)[0][0]
yshift_diff_frame = np.where(np.abs(yshift_Parallel_zpos - yshift_new_Parallel_zpos) > 0)[0][0]
print(f'First frame where xshift_Parallel_zpos and xshift_new_Parallel_zpos differ: {xshift_diff_frame}')
print(f'First frame where yshift_Parallel_zpos and yshift_new_Parallel_zpos differ: {yshift_diff_frame}')

In [ ]:
# Test shift code with made-up zcorr and shift values

from tifffile import TiffFile
from caiman.mmapping import load_memmap
import time
import concurrent.futures
from suite2p.registration import rigid
from scipy.ndimage import gaussian_filter
 
def makeup_zcorrel(zstack_file, movie_mmap_path, shift_in_x=None, shift_in_y=None):
    """
    Compute z-position and x/y shifts of each frame in the movie.
    """
    ops = cz.default_zcorr_params()
    ops["nonrigid"] = False
    
    # Load the z-stack and stack frames into a 3D array
    zstack_tiff_file = TiffFile(zstack_file)
    Zreg = []
    for page in zstack_tiff_file.pages:
        Zreg.append(page.asarray())
    Zreg = np.stack(Zreg, axis=0)
    zstack_tiff_file.close()
    
    # Flip Zreg to match the orientation of the movie
    Zreg = np.flip(Zreg, axis=1)

    # Load the movie
    movie_16bit, dims, T = load_memmap(movie_mmap_path)
    Treg = np.reshape(movie_16bit.T, [T] + list(dims), order='F')
    
    # Get the size of the movie
    Ly, Lx = Treg.shape[1], Treg.shape[2]
    
    # Crop Zreg to match the size of the movie
    if Zreg.shape[1] > Ly or Zreg.shape[2] != Lx:
        Zreg = Zreg[:, :Ly, :Lx]
        
    # Shift the z-stack according to the x/y shifts
    if shift_in_x is not None and shift_in_y is not None:
        if isinstance(shift_in_x, int) and isinstance(shift_in_y, int):
            stack_shift_in_x = [shift_in_x] * Zreg.shape[0]
            stack_shift_in_y = [shift_in_y] * Zreg.shape[0]
        else:
            stack_shift_in_x = shift_in_x
            stack_shift_in_y = shift_in_y
        Zreg = np.stack([np.roll(Z, (y, x), axis=(0, 1)) for Z, x, y in zip(Zreg, stack_shift_in_x, stack_shift_in_y)], axis=0)
    
    # Get the number of frames in the movie
    nFrames, _, _ = Treg.shape
    
    # Initialize arrays
    yshift = np.zeros((Zreg.shape[0], nFrames), np.int32)
    xshift = np.zeros((Zreg.shape[0], nFrames), np.int32)
    zcorr = np.zeros((Zreg.shape[0], nFrames), np.float32)
    
    t0 = time.time()
    refAndMasks = []
    for Z in Zreg:
        maskMul, maskOffset = rigid.compute_masks(
            refImg=Z,
            maskSlope=3 * ops["smooth_sigma"],
        )
        # Compute the cross-correlation reference image
        cfRefImag = rigid.phasecorr_reference(refImg=Z, smooth_sigma=ops["smooth_sigma"])
        # Add a new axis to cfRefImag
        cfRefImag = cfRefImag[np.newaxis, :, :]
        # Append the maskMul, maskOffset, and cfRefImag to refAndMasks
        refAndMasks.append((maskMul, maskOffset, cfRefImag))

    # Compute zcorr for each frame
    with concurrent.futures.ThreadPoolExecutor() as executor:
        # Submit the compute_zcorr_for_frame function to the executor
        futures = [executor.submit(cz.compute_zcorr_for_frame, nfr, Treg, refAndMasks, ops) for nfr in range(nFrames)]
        # As the threads complete, store the results in the yshift, xshift, and zcorr arrays
        for i, future in enumerate(concurrent.futures.as_completed(futures)):
            # zcorr[:, i] = future.result().squeeze()
            yshift_res, xshift_res, zcorr_res = future.result()
            yshift[:, i], xshift[:, i], zcorr[:, i] = (
                np.asarray(yshift_res, dtype=np.int32).squeeze(),
                np.asarray(xshift_res, dtype=np.int32).squeeze(),
                np.asarray(zcorr_res, dtype=np.float32).squeeze()
            )
            # yshift[:, i], xshift[:, i], zcorr[:, i] = map(lambda x: np.asarray(x, dtype=np.float32).squeeze(), future.result())            
            if i % 10 == 0:
                print(f"{i + 1}/{nFrames} frames, {time.time() - t0:.2f} sec.")
                                
    print(f"{nFrames}/{nFrames} frames, {time.time() - t0:.2f} sec.")
                                    
    # Get the optimal z position for each frame
    zpos = np.argmax(gaussian_filter(zcorr, sigma=3, axes=0), axis=0)
    zpos = zpos.astype(np.uint16)
    
    # Save z-correlation variables to a file in the mesmerize folder, preserving data types and accuracy
    z_correlation = {
        'yshift': yshift.astype(np.int32),
        'xshift': xshift.astype(np.int32),
        'zcorr': zcorr.astype(np.float32),
        'zpos': zpos.astype(np.uint16)
    }

    # Save the z-correlation variables to a file in the same directory as the movie, with the shift values in the name
    np.savez(movie_mmap_path.parents[1] / f"z_correlation_mu_{shift_in_x}_{shift_in_y}.npz", **z_correlation)
    # Save the Z-stack to a file in the same directory as the movie, with the shift values in the name
    np.save(movie_mmap_path.parents[1] / f"Zreg_mu_{shift_in_x}_{shift_in_y}.npy", Zreg)

    return z_correlation, Zreg

In [ ]:
# To test code with made-up shifted values, uncomment lines below (and all containing "_mu" variables after)  
# # Generate made-up zcorr and shift values
# shift_in_x = 5
# shift_in_y = 10

# if not os.path.exists(mesmerize_path / f"z_correlation_mu_{shift_in_x}_{shift_in_y}.npz"):
#     z_correlation_mu, Zstack_mu=makeup_zcorrel(zstack_path / z_shifted_file, mcorr_movie_path, shift_in_x, shift_in_y)
# else:
#     z_correlation_mu=np.load(mesmerize_path / f"z_correlation_mu_{shift_in_x}_{shift_in_y}.npz")
#     Zstack_mu=np.load(mesmerize_path / f"Zreg_mu_{shift_in_x}_{shift_in_y}.npy")

# # Print the first 10 shifts corresponding to the first 10 frames' zpos
# print(z_correlation_mu['xshift'][z_correlation_mu['zpos'][0], :10])
# print(z_correlation_mu['yshift'][z_correlation_mu['zpos'][0], :10])
# # Shifts should be the opposite of the shift_in_x and shift_in_y values (added to the "inherent" shift in the z-stack)

In [ ]:
# z_pos = np.load(mesmerize_path / "z_correlation_old.npy")
# z_pos = z_correlation
# z_pos.shape
# z_correlation_saved = np.load(mesmerize_path / "z_correlation.npz")

In [ ]:
# from scipy.ndimage import gaussian_filter
# z_corr = z_correlation['zcorr']
# #  Print first column of z_corr
# print(z_corr[:,0])
# print(np.argmax(z_corr[:,0]))
# print(gaussian_filter(z_corr[:,0], sigma=3, axes=0))
# print(np.argmax(gaussian_filter(z_corr[:,0], sigma=3, axes=0)))

In [ ]:
# from scipy.ndimage import gaussian_filter
# zpos = np.argmax(gaussian_filter(z_correlation_saved['zcorr'], sigma=3, axes=0), axis=0)

In [ ]:
# Plot z-correlation
import matplotlib.pyplot as plt
plt.figure(figsize=(15, 5))
plt.plot(z_correlation['zpos'], label='zpos')    
# plt.plot(z_correlation_mu['zpos'], label='zpos_mu')
plt.xlabel('T-series Frame')
plt.ylabel('Z-stack Frame')
plt.title('Z-correlation')
plt.show()

In [ ]:
from caiman.mmapping import load_memmap
from tqdm import tqdm
from scipy.stats import mode
from scipy.ndimage import gaussian_filter, affine_transform
from skimage.transform import warp
from skimage.registration import phase_cross_correlation
from PIL import Image
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import multiprocessing

# Load the movie
movie_16bit, dims, T = load_memmap(mcorr_movie_path)
pixels = np.reshape(movie_16bit.T, [T] + list(dims), order='F')
Nframe, Ny, Nx = pixels.shape
# Clip the movie range to uint16
pixels = np.clip(pixels, 0, 2**16-1)

# Load the FOV file
# mcorr_batch_dir = mcorr_movie_path.parent
# fov_file_path = Path.joinpath(mcorr_batch_dir, f"{mcorr_batch_dir.stem}_mean_projection.npy")
# fov_image = np.load(fov_file_path)

# Load the z-stack
info_zstack = Image.open(shift_zstack_path)
Nz = info_zstack.n_frames
Zstack = np.zeros((Ny, Nx, Nz), dtype=np.float32)
for iz in range(Nz):
    info_zstack.seek(iz)
    Zstack[:, :, iz] = np.array(info_zstack)
info_zstack.close()        
# TODO: check why on Windows F_anat bit depth is 8 and not 16

# Flip Zstack up-down to match the orientation of the movie
Zstack = np.flip(Zstack, axis=0)

In [ ]:
# Print min max values of the movie and zstack
print(f"Movie min: {pixels.min()}, max: {pixels.max()}")
print(f"Zstack min: {Zstack.min()}, max: {Zstack.max()}")
# Print data types
print(f"Movie data type: {pixels.dtype}")
print(f"Zstack data type: {Zstack.dtype}")

In [ ]:
# Plot the first image of the movie, the FOV image, and the top slice of the z-stack side by side
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].imshow(pixels[0], cmap='gray')
ax[0].set_title("Movie")
# ax[1].imshow(fov_image, cmap='gray')
# ax[1].set_title("FOV")
ax[1].imshow(Zstack[:, :, -1], cmap='gray')
ax[1].set_title("Z-stack")
plt.show()

In [ ]:
print("Zstack shape:", Zstack.shape)
# Move Zstack_mu's dimensions to match Zstack's
# Zstack_mu = np.moveaxis(Zstack_mu, 0, -1)
# print("Zstack_mu shape:", Zstack_mu.shape)


In [ ]:
zpos = z_correlation['zpos']
# zpos = z_correlation_mu['zpos']
# Check if zpos +/-2 are valid indices
if (zpos-2 < 0).any() or (zpos+2 >= Zstack.shape[2]).any():
    raise ValueError("Z-stack does not have enough frames for zcorr")

### F_anat movie
Tests also include making a F_anat movie. This is not needed in final code. Change `use_f_anat` to `True` below if using. Other, skip to [Correlate motion-corrected F functional with Z-stack](#z_correl_step). 


In [ ]:
use_f_anat = False

In [ ]:
if use_f_anat:
    # Make a 5-layer movie of the z-stack frames that follows zcorr (movement in depth).
    # Middle layer is zpos, 2 layers above and 2 layers below are zpos+1 and +2 and zpos-1 and -2, respectively.
    F_anat = np.stack([Zstack[:, :, zpos-2], Zstack[:, :, zpos-1], Zstack[:, :, zpos], Zstack[:, :, zpos+1], Zstack[:, :, zpos+2]], axis=-1)
    # F_anat_mu = np.stack([Zstack_mu[:, :, zpos-2], Zstack_mu[:, :, zpos-1], Zstack_mu[:, :, zpos], Zstack_mu[:, :, zpos+1], Zstack_mu[:, :, zpos+2]], axis=-1)

In [ ]:
if use_f_anat:
    # Now we apply the xshift and yshift, computed earlier between the mcorr_movie and the z-stack, to the z-stack frames in F_anat

    # Create the corresponding array to store xshift yshift for each zpos, across frames
    xshift = z_correlation['xshift']
    yshift = z_correlation['yshift']
    # xshift = z_correlation_mu['xshift']
    # yshift = z_correlation_mu['yshift']

    # Initialize F_anat_shifted with the same shape as F_anat
    F_anat_shifted = np.zeros_like(F_anat)
    # F_anat_shifted = np.zeros_like(F_anat_mu)

    # Shift F_anat with xshift and yshift
    for frameNum in range(Nframe):
        current_zpos = zpos[frameNum]
        indices = [current_zpos - 2, current_zpos - 1, current_zpos, current_zpos + 1, current_zpos + 2]
        
        # Ensure all indices are within the valid range
        valid_indices = [index for index in indices if 0 <= index < Zstack.shape[2]]
        layer_mapping = {index: idx for idx, index in enumerate(indices) if index in valid_indices}

        for idx in valid_indices:
            # Get the x and y shift for the current frame and zpos, and reverse the direction.
            frameXshift = -xshift[idx, frameNum]
            frameYshift = -yshift[idx, frameNum]
            # Shift the corresponding layer of F_anat
            layer_index = layer_mapping[idx]
            if 0 <= layer_index < F_anat.shape[3]:  # Check if the layer index is valid for F_anat
            # if 0 <= layer_index < F_anat_mu.shape[3]:   
                            
                # Use affine transformation to shift the layer
                transformation_matrix = np.array([[1, 0, frameYshift], [0, 1, frameXshift]])
                F_anat_shifted[:, :, frameNum, layer_index] = affine_transform(F_anat[:, :, frameNum, layer_index],
                                                                            transformation_matrix,
                                                                            order=1,
                                                                            mode='constant',
                                                                            cval=0.0,
                                                                            prefilter=False)
                # F_anat_shifted[:, :, frameNum, layer_index] = affine_transform(F_anat_mu[:, :, frameNum, layer_index],
                #                                                                transformation_matrix,
                #                                                                order=1,
                #                                                                mode='constant',
                #                                                                cval=0.0,
                #                                                                prefilter=False)
                

In [ ]:
if use_f_anat:
    # Plot the 6th middle frame of F_anat and F_anat_shifted
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(F_anat[:, :, 5, 2].squeeze(), cmap='gray')
    # ax[0].imshow(F_anat_mu[:, :, 5, 2].squeeze(), cmap='gray')
    ax[0].set_title("F_anat")
    ax[1].imshow(F_anat_shifted[:, :, 5, 2].squeeze(), cmap='gray')
    ax[1].set_title("F_anat_shifted")
    plt.show()

In [ ]:
if use_f_anat:
    # save the first 50 middle frames (corresponding to zpos) of F_anat and F_anat_shifted to a movie file
    from tifffile import imwrite
    F_anat_excerpt = F_anat[:, :, 0:50, 2].squeeze()
    # F_anat_excerpt = F_anat_mu[:, :, 0:50, 2].squeeze()
    F_anat_shifted_excerpt = F_anat_shifted[:, :, 0:50, 2].squeeze()
    # print the shapes of the two arrays
    print(F_anat_excerpt.shape)
    print(F_anat_shifted_excerpt.shape)
    #  plot the first frame of the two arrays
    # fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    # ax[0].imshow(F_anat_excerpt[:, :, 0], cmap='gray')
    # ax[0].set_title("F_anat_excerpt")
    # ax[1].imshow(F_anat_shifted_excerpt[:, :, 0], cmap='gray')
    # ax[1].set_title("F_anat_shifted_excerpt")
    # plt.show()
    #  Concatenate the two horizontally
    F_anat_concatenated = np.concatenate([F_anat_excerpt, F_anat_shifted_excerpt], axis=1)
    print(F_anat_concatenated.shape)
    # plt.imshow(F_anat_concatenated[:, :, 0], cmap='gray')
    # plt.show()
    # Reorder dimension to (T, H, W) for tiff file
    F_anat_concatenated = np.transpose(F_anat_concatenated, (2, 0, 1))
    print(F_anat_concatenated.shape)
    # save to avi file
    imwrite(mesmerize_path / 'F_anat_F_anat_shifted.tiff', F_anat_concatenated)

In [ ]:
# if use_f_anat:
    # # Overlay the diff between F_anat and F_anat_shifted on F_anat as red color with transparency
    # diff = np.abs(F_anat_excerpt - F_anat_shifted_excerpt)

    # # Scale the difference to the range of the original data if not already [0, 255]
    # diff_scaled = (diff * 255 / np.max(diff)).astype(np.uint8)

    # # Create an RGBA image for the overlay where red channel highlights the difference
    # overlay = np.zeros((*F_anat_excerpt.shape, 4), dtype=np.uint8)  # Shape: (height, width, 4 for RGBA)
    # # overlay[..., 1] = diff_scaled  # Green channel
    # overlay[..., 0] = diff_scaled  # Red channel
    # overlay[..., 3] = (diff_scaled * 1).astype(np.uint8)  # Alpha channel. 0.5 for 50% transparency

    # # Convert F_anat_excerpt to grayscale image suitable for RGB representation
    # F_anat_rgb = np.stack([F_anat_excerpt]*3, axis=-1).astype(np.uint8)

    # # Normalize F_anat_rgb to [0, 1]
    # F_anat_rgb_normalized = F_anat_rgb / 65535.0

    # # Plot using Matplotlib to visualize the overlay
    # plt.figure(figsize=(10, 8))
    # plt.imshow(F_anat_excerpt[:, :, 6], cmap='gray')
    # plt.imshow(overlay[:, :, 6, :], interpolation='none')
    # plt.axis('off')
    # plt.show()

    # Merge the overlay with the original excerpt to save it as an image
    # Flatten the overlay onto the RGB excerpt by blending based on alpha
    # alpha = overlay[..., 3] / 255.0
    # for c in range(3):  # Only RGB channels, not alpha
    #     F_anat_rgb[..., c] = (1 - alpha) * F_anat_rgb[..., c] + alpha * overlay[..., c]

    # # Reorder dimensions to (T, H, W, C) for tiff file
    # F_anat_rgb = np.transpose(F_anat_rgb, (2, 0, 1, 3))

    # # # Save the merged output as a TIFF file
    # imwrite(mesmerize_path /'F_anat_with_diff_overlay.tif', F_anat_rgb)

In [ ]:
if use_f_anat:
    # print range of F_anat_excerpt and F_anat_shifted_excerpt
    print(f"F_anat_excerpt min: {F_anat_excerpt.min()}, max: {F_anat_excerpt.max()}")
    print(f"F_anat_shifted_excerpt min: {F_anat_shifted_excerpt.min()}, max: {F_anat_shifted_excerpt.max()}")

In [ ]:
# if use_f_anat:
    # # Example shifts
    # shift_y, shift_x = 5, 0  # Shift 10 pixels down and 15 pixels right

    # # Create transformation matrix for shifting
    # # Note: scipy.ndimage affine_transform uses matrix inverses compared to usual definitions
    # transformation_matrix = np.array([[1, 0, -shift_y],
    #                                   [0, 1, -shift_x],
    #                                   [0, 0, 1]])

    # F_anat_frame= F_anat_excerpt[:,:,0]

    # # Apply transformation
    # F_anat_frame_shifted = affine_transform(F_anat_frame, transformation_matrix[:2, :2], offset=transformation_matrix[:2, 2], 
    #                                   output_shape=F_anat_frame.shape, cval=0.0, order=1)

    # # Do the same with np.roll
    # # F_anat_frame_shifted = np.roll(F_anat_frame, (shift_y, shift_x), axis=(0, 1)) 

    # F_anat_shifted_norm = F_anat_frame_shifted / 65535.0
    # F_anat_norm = F_anat_frame / 65535.0

    # # Create an RGBA image for the backgroung where red channel is emphasized
    # backgroung = np.zeros((*F_anat_norm.shape, 4), dtype=np.float32)  # Using float32 for better precision with normalized data
    # backgroung[..., 0] = F_anat_norm  # Red channel from normalized shifted data
    # backgroung[..., 3] = 1  # Set a constant. 0.5 for 50% transparency

    # # Create an RGBA image for the overlay where green channel is emphasized
    # overlay = np.zeros((*F_anat_shifted_norm.shape, 4), dtype=np.float32)  # Using float32 for better precision with normalized data
    # overlay[..., 1] = F_anat_shifted_norm  # Green channel from normalized shifted data
    # overlay[..., 3] = 0.5  # Set a constant 50% transparency across the overlay

    # # Display background and overlay
    # plt.figure(figsize=(10, 8))
    # plt.imshow(backgroung, interpolation='none') 
    # plt.imshow(overlay, interpolation='none')
    # plt.axis('off')  # Hide axes
    # plt.show()

In [ ]:
if use_f_anat:
    # Overlay F_anat_shifted_excerpt over F_anat_excerpt as green color with transparency
    # Normalize the excerpts to [0, 1] for display purposes
    F_anat_shifted_norm = F_anat_shifted_excerpt / 65535.0
    F_anat_norm = F_anat_excerpt / 65535.0

    # Create an RGBA image for the backgroung where red channel is emphasized
    backgroung = np.zeros((*F_anat_norm.shape, 4), dtype=np.float32)  # Using float32 for better precision with normalized data
    backgroung[..., 0] = F_anat_norm  # Green channel from normalized shifted data
    backgroung[..., 3] = 1  # Set a constant. 0.5 for 50% transparency

    # Create an RGBA image for the overlay where green channel is emphasized
    overlay = np.zeros((*F_anat_shifted_norm.shape, 4), dtype=np.float32)  # Using float32 for better precision with normalized data
    overlay[..., 1] = F_anat_shifted_norm  # Green channel from normalized shifted data
    overlay[..., 3] = 0.5  # Set a constant 50% transparency across the overlay

    # # Plot the original image in grayscale
    plt.figure(figsize=(10, 8))
    plt.imshow(backgroung[:,:,0,:], interpolation='none') 
    # Overlay the green-shifted image
    plt.imshow(overlay[:,:,0,:], interpolation='none')
    plt.axis('off')  # Hide axes
    plt.show()

    print(f"shift_in_x: {shift_in_x}, shift_in_y: {shift_in_y}")
    # Cells should overlap (= be yellow), unless plotting the shifted F_anat,
    # computed with mu shifts on the original F_anat (not F_anat_mu), overlayed the original F_anat.

In [ ]:
rf = parameters['params_extraction']['main']['rf']
rf

### Correlate motion-corrected F functional with Z-stack
* Apply shifts computed earlier
* Use z-pos +/- 2 px in depth
* Compute correlation over patches

In [ ]:
# import numpy as np
# import scipy.stats as stats
# import pandas as pd

# # Set dimensions
# Nframe, Ny, Nx = pixels.shape
# # Zstack_shape = F_anat.shape
# Zstack_shape = Zstack.shape

# # Define patch size based on Caiman's parameters. Width of patch = strides+overlaps.
# mcorr_params = parameters['params_mcorr']['main']
# step_size = mcorr_params['strides'] 
# patch_overlap = mcorr_params['overlaps']
# # Calculate patch size for both x and y dimensions 
# patch_size = [step + overlap for step, overlap in zip(step_size, patch_overlap)]

# # Get x and y shifts
# xshift = z_correlation['xshift']
# yshift = z_correlation['yshift']

# # Initialize a list to store correlation coefficients
# patch_correlations = []
# # patch_correlations = pd.DataFrame(columns=['r_squared', 'slope', 'intercept',
# #                                          'frameNum', 'patch_x_idx', 'patch_y_idx', 'patch_z_idx'])

# # Iterate over each frame in the T-series
# for frameNum in range(Nframe):
#     current_zpos = zpos[frameNum]
#     indices = [current_zpos - 3, current_zpos - 2, current_zpos - 1,
#                current_zpos, current_zpos + 1, current_zpos + 2, current_zpos + 3]
    
#     # Ensure all indices are within the valid range
#     valid_indices = [index for index in indices if 0 <= index < Zstack_shape[2]]
    
#     Z_frame_shifted = np.zeros_like(Zstack)
    
#     # Check against each z-slice in the anatomical stack
#     # for z in range(5):  # 5 layers in z-stack around zpos
#     for z_idx in valid_indices:
#         # Get the x and y shift for the current frame and zpos, and reverse the direction.
#         frameXshift = -xshift[z_idx, frameNum]
#         frameYshift = -yshift[z_idx, frameNum]
        
#         # Shift the corresponding layer of Zstack, using affine transformation
#         transformation_matrix = np.array([[1, 0, frameYshift], [0, 1, frameXshift]])
#         Z_frame_shifted[:, :, z_idx] = affine_transform(Zstack[:, :, z_idx],
#                                                         transformation_matrix,
#                                                         order=1,
#                                                         mode='constant',
#                                                         cval=0.0,
#                                                         prefilter=False)
        
#     for i in range(0, Nx - patch_size[0] + 1, step_size[0]):
#         for j in range(0, Ny - patch_size[1] + 1, step_size[1]):
#             # Get F func patch and flatten it. pixels dimensions are frame, x, y.
#             T_patch = pixels[frameNum, i:i+patch_size[0], j:j+patch_size[1]].ravel()
            
#             # Patch number
#             patch_num = i // step_size[0] * (Ny // step_size[1]) + j // step_size[1]
            
#             for z_idx in valid_indices:
#                 # Get F anat patch from the z-stack. Zstack dimensions are x, y, z.
#                 Z_patch = Z_frame_shifted[i:i+patch_size[0], j:j+patch_size[1], z_idx].ravel()
#                 if T_patch.size == Z_patch.size:  # Ensure matching patch sizes
#                     slope, intercept, r_value, p_value, std_err = stats.linregress(T_patch, Z_patch)

#                     # Collect data into a list of dictionaries
#                     patch_correlations.append({
#                         'r_squared': r_value**2, 'slope': slope, 'intercept': intercept,
#                         'frame_num': frameNum, 'patch_number': patch_num, 'patch_x_lims': [i,i+patch_size[0]], 'patch_y_lims': [j,j+patch_size[1]], 'patch_z_idx': z_idx - current_zpos, 'patch_z_pos': z_idx
#                     })

# # Convert list of dictionaries to DataFrame
# patch_correlations_df = pd.DataFrame(patch_correlations)

# # Executed in 9m 40s on nw-02

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm
# from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
import numpy as np
import scipy.stats as stats
import pandas as pd
from scipy.ndimage import affine_transform

np.seterr(over='raise')

# Time the execution
import time
start_time = time.time()

def process_frame(frame_data):
    fram_start_time = time.time()
    
    frameNum, frame_data, Zstack, frame_xshift, frame_yshift, Ny, Nx, patch_size, step_size, current_zpos, valid_indices = frame_data

    Z_frame_shifted = np.zeros_like(Zstack)
    results = []
    
    for z_idx in valid_indices:
        frameXshift = -frame_xshift[z_idx]
        frameYshift = -frame_yshift[z_idx]
        transformation_matrix = np.array([[1, 0, frameYshift], [0, 1, frameXshift]])
        Z_frame_shifted[:, :, z_idx] = affine_transform(Zstack[:, :, z_idx], transformation_matrix, order=1, mode='constant', cval=0.0, prefilter=False)
    
    for i in range(0, Nx - patch_size[0] + 1, step_size[0]):
        for j in range(0, Ny - patch_size[1] + 1, step_size[1]):
            T_patch = frame_data[i:i+patch_size[0], j:j+patch_size[1]].ravel()
            patch_num = i // step_size[0] * (Ny // step_size[1]) + j // step_size[1]
            
            for z_idx in valid_indices:
                Z_patch = Z_frame_shifted[i:i+patch_size[0], j:j+patch_size[1], z_idx].ravel()
                if T_patch.size == Z_patch.size:
                    slope, intercept, r_value, p_value, std_err = stats.linregress(T_patch, Z_patch)
                    try:
                        results.append({
                            'r_squared': r_value**2, 'slope': slope, 'intercept': intercept,
                            'frame_num': frameNum, 'patch_number': patch_num, 'patch_x_lims': [i, i+patch_size[0]], 'patch_y_lims': [j, j+patch_size[1]], 'patch_z_idx': int(z_idx) - int(current_zpos), 'patch_z_pos': z_idx
                        })
                    # try:
                    #     print({
                    #         "r_squared": r_value**2, 
                    #         "slope": slope, 
                    #         "intercept": intercept,
                    #         "frame_num": frameNum, 
                    #         "patch_number": patch_num, 
                    #         "patch_x_lims": [i, i+patch_size[0]], 
                    #         "patch_y_lims": [j, j+patch_size[1]], 
                    #         "patch_z_idx": int(z_idx) - int(current_zpos), 
                    #         "patch_z_pos": z_idx
                    #     })
                    except FloatingPointError:
                        print("Overflow Error")
                        print(f"z_idx: {z_idx}, current_zpos: {current_zpos}")
                        
    frame_end_time = time.time()
    print(f"Frame {frameNum} processed in {frame_end_time - fram_start_time:.2f} seconds")
            
    return results

# Set dimensions
Nframe, Ny, Nx = pixels.shape
# Zstack_shape = F_anat.shape
Zstack_shape = Zstack.shape

# Define patch size based on Caiman's parameters. Width of patch = strides+overlaps.
mcorr_params = parameters['params_mcorr']['main']
step_size = mcorr_params['strides'] 
patch_overlap = mcorr_params['overlaps']
# Calculate patch size for both x and y dimensions 
patch_size = [step + overlap for step, overlap in zip(step_size, patch_overlap)]

# Get x and y shifts
xshift = z_correlation['xshift']
yshift = z_correlation['yshift']

# Prepare data for parallel processing
frame_data_list = []
for frameNum in range(Nframe):
    current_zpos = zpos[frameNum]
    indices = [current_zpos - 3, current_zpos - 2, current_zpos - 1,
               current_zpos, current_zpos + 1, current_zpos + 2, current_zpos + 3]
    valid_indices = [index for index in indices if 0 <= index < Zstack_shape[2]]
    
    # Prepare data for each frame to be processed in parallel
    # frame_data_list = [(frameNum, pixels[frameNum, :, :], Zstack, xshift[:, frameNum], yshift[:, frameNum], Ny, Nx, patch_size, step_size, zpos[frameNum], valid_indices) for frameNum in range(Nframe)]

    frame_data_list.append((frameNum, pixels[frameNum, :, :], Zstack, xshift[:, frameNum], yshift[:, frameNum], Ny, Nx, patch_size, step_size, zpos[frameNum], valid_indices))

# for frame_data in frame_data_list:
#     results = process_frame(frame_data)

# Use ProcessPoolExecutor to parallelize the task
# with ProcessPoolExecutor() as executor:
# # # Use ThreadPoolExecutor to parallelize the task
# # with ThreadPoolExecutor() as executor:
#     futures = {executor.submit(process_frame, frame_data): frame_data for frame_data in frame_data_list}
    
#     results = []
#     for future in tqdm(as_completed(futures), total=len(futures), desc="Processing frames"):
#         results.append(future.result())
        
# Use Parallel from joblib to parallelize the task 
results = Parallel(n_jobs=-1)(delayed(process_frame)(frame_data) for frame_data in frame_data_list)
results = Parallel(n_jobs=-1)(delayed(process_frame)(frame_data) for frame_data in tqdm(frame_data_list, desc="Processing frames"))

# Flatten list of lists returned by map
patch_correlations = [item for sublist in results for item in sublist]

# Convert to DataFrame
patch_correlations_df = pd.DataFrame(patch_correlations)

# Display execution time in mn and seconds
exect_time = time.time() - start_time
print("Correlation computation took: ", exect_time // 60, "mn and ", exect_time % 60, "seconds")

# ThreadPoolExecutor correlation computation took:  18 mn 22 s
# ProcessPoolExecutor correlation computation took:  7 mn 53 s
# Parallel correlation computation took:  1 mn 10 s

In [ ]:
# # Save the patch_correlations_df to a table in a .mat file
# from scipy.io import savemat
# # Convert DataFrame to a dictionary
# data_dict = {col: patch_correlations_df[col].values for col in patch_correlations_df.columns}
# savemat(mesmerize_path / 'patch_correlations_dict.mat', {'patch_correlations': data_dict})
# Save the patch_correlations_df to a csv file
patch_correlations_df.to_csv(mesmerize_path / 'patch_correlations_ProcessPoolExecutor.csv', index=False)

In [ ]:
patch_correlations_df.shape

In [ ]:
# Compute fits
# Reshape the "anatomical" fluorescence movie
F_anat = F_anat.reshape(Ny * Nx, Nframe)

# Get the average image of the z-stack movie
F_anat_mean = np.mean(F_anat, axis=1)